# NYC Mobility - Great Expectations Quality Gate

This notebook validates the NYC Mobility pipeline using Great Expectations 1.x with PySpark.

The quality gate evaluates:
- Consistency
- Completeness
- Uniqueness
- Validity
- Timeliness and volume
- Accuracy scope
- Auditability

The existing DQ logic is calculated using PySpark, while Great Expectations is used to validate that all governed checks have a `PASS` status.

The pipeline fails if any quality check fails.

#### Python: Imports and GX setup

In [0]:
%pip install great_expectations

In [0]:
import great_expectations as gx
from pyspark.sql import functions as F

# Great Expectations context
context = gx.get_context()

## Create Quality Check Results

Each check records:

- `check_name`: name of the validation
- `quality_attribute`: quality dimension being evaluated
- `expected_value`: expected condition
- `actual_value`: calculated result
- `status`: `PASS` or `FAIL`

These checks reproduce the logic of the existing NYC Mobility quality gate.

In [0]:
from pyspark.sql import functions as F

bronze_green = spark.table("`ftw-week-08`.`01_bronze`.green_taxi")
silver_green = spark.table("`ftw-week-08`.`02_silver`.green_taxi")
gold_fact = spark.table("`ftw-week-08`.`03_gold`.fact_green_taxi_trip")
silver_zones = spark.table("`ftw-week-08`.`02_silver`.taxi_zones")
gold_zone = spark.table("`ftw-week-08`.`03_gold`.dim_taxi_zone")
silver_weather = spark.table("`ftw-week-08`.`02_silver`.weather")
gold_weather = spark.table("`ftw-week-08`.`03_gold`.dim_weather_hour")
gold_date = spark.table("`ftw-week-08`.`03_gold`.dim_date")
gold_time = spark.table("`ftw-week-08`.`03_gold`.dim_time")
dq_referential = spark.table(
    "`ftw-week-08`.`03_gold`.dq_dashboard_referential_integrity"
)
analytics_validation = spark.table(
    "`ftw-week-08`.`03_gold`.analytics_validation_summary"
)
dq_dimensions = spark.table(
    "`ftw-week-08`.`03_gold`.dq_dashboard_canonical_dimensions"
)

checks = []

# 1. Bronze to Silver Green Taxi row preservation

bronze_count = bronze_green.count()
silver_count = silver_green.count()

checks.append((
    "Bronze to Silver Green Taxi row preservation",
    "CONSISTENCY",
    str(bronze_count),
    str(silver_count),
    "PASS" if bronze_count == silver_count else "FAIL"
))


# 2. Silver to Gold fact row preservation

gold_fact_count = gold_fact.count()

checks.append((
    "Silver to Gold fact row preservation",
    "CONSISTENCY",
    str(silver_count),
    str(gold_fact_count),
    "PASS" if silver_count == gold_fact_count else "FAIL"
))


# 3. Silver to Gold retained measures reconcile

measure_columns = [
    "trip_distance",
    "trip_duration_minutes",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "congestion_surcharge",
    "cbd_congestion_fee",
    "total_amount"
]

silver_measures = silver_green.select([
    F.sum(F.col(c)).alias(c)
    for c in measure_columns
]).first()

gold_measures = gold_fact.select([
    F.sum(F.col(c)).alias(c)
    for c in measure_columns
]).first()

measure_difference = sum(
    abs(
        float(silver_measures[c] or 0)
        - float(gold_measures[c] or 0)
    )
    for c in measure_columns
)

measure_difference = round(measure_difference, 6)

checks.append((
    "Silver to Gold retained measures reconcile",
    "CONSISTENCY",
    "0 aggregate difference",
    f"{measure_difference} aggregate difference",
    "PASS" if measure_difference == 0 else "FAIL"
))


# 4. Silver to Gold Taxi Zone row preservation

silver_zone_count = silver_zones.count()
gold_zone_count = gold_zone.count()

checks.append((
    "Silver to Gold Taxi Zone row preservation",
    "CONSISTENCY",
    str(silver_zone_count),
    str(gold_zone_count),
    "PASS" if silver_zone_count == gold_zone_count else "FAIL"
))


# 5. Gold Weather contains Silver hours plus one Unknown

silver_weather_count = silver_weather.count()
gold_weather_count = gold_weather.count()

expected_weather_count = silver_weather_count + 1

checks.append((
    "Gold Weather contains Silver hours plus one Unknown member",
    "COMPLETENESS",
    str(expected_weather_count),
    str(gold_weather_count),
    "PASS" if gold_weather_count == expected_weather_count else "FAIL"
))


# 6. Gold dimension keys are unique

null_date_keys = gold_date.filter(F.col("date_key").isNull()).count()
null_time_keys = gold_time.filter(F.col("time_key").isNull()).count()
null_zone_keys = gold_zone.filter(F.col("taxi_zone_key").isNull()).count()
null_weather_keys = gold_weather.filter(F.col("weather_hour_key").isNull()).count()

dup_date = (
    gold_date.filter(F.col("date_key").isNotNull()).count()
    - gold_date.filter(F.col("date_key").isNotNull()).select(F.countDistinct("date_key")).first()[0]
)
dup_time = (
    gold_time.filter(F.col("time_key").isNotNull()).count()
    - gold_time.filter(F.col("time_key").isNotNull()).select(F.countDistinct("time_key")).first()[0]
)
dup_zone = (
    gold_zone.filter(F.col("taxi_zone_key").isNotNull()).count()
    - gold_zone.filter(F.col("taxi_zone_key").isNotNull()).select(F.countDistinct("taxi_zone_key")).first()[0]
)
dup_weather = (
    gold_weather.filter(F.col("weather_hour_key").isNotNull()).count()
    - gold_weather.filter(F.col("weather_hour_key").isNotNull()).select(F.countDistinct("weather_hour_key")).first()[0]
)

total_null_keys = null_date_keys + null_time_keys + null_zone_keys + null_weather_keys
total_dup_keys = dup_date + dup_time + dup_zone + dup_weather
total_key_issues = total_null_keys + total_dup_keys

checks.append((
    "Gold dimension keys are unique",
    "UNIQUENESS",
    "0 duplicate or null-key rows",
    str(total_key_issues),
    "PASS" if total_key_issues == 0 else "FAIL"
))


# 7. Gold fact technical keys are complete and unique

null_trip_keys = gold_fact.filter(F.col("trip_key").isNull()).count()
non_null_trip_count = gold_fact.filter(F.col("trip_key").isNotNull()).count()
dup_trip_key = (
    non_null_trip_count
    - gold_fact.filter(F.col("trip_key").isNotNull()).select(F.countDistinct("trip_key")).first()[0]
)
total_trip_key_issues = null_trip_keys + dup_trip_key

checks.append((
    "Gold fact technical keys are complete and unique",
    "UNIQUENESS",
    "0 null or duplicate-key rows",
    str(total_trip_key_issues),
    "PASS" if total_trip_key_issues == 0 else "FAIL"
))


# 8. Fact foreign keys resolve without join multiplication

ri_row = dq_referential.first()
total_missing = (
    int(ri_row["missing_pickup_date_keys"] or 0)
    + int(ri_row["missing_dropoff_date_keys"] or 0)
    + int(ri_row["missing_pickup_time_keys"] or 0)
    + int(ri_row["missing_dropoff_time_keys"] or 0)
    + int(ri_row["missing_pickup_zone_keys"] or 0)
    + int(ri_row["missing_dropoff_zone_keys"] or 0)
    + int(ri_row["missing_weather_keys"] or 0)
)
join_diff = abs(int(ri_row["join_row_difference"] or 0))
ri_total = total_missing + join_diff

checks.append((
    "Fact foreign keys resolve without join multiplication",
    "CONSISTENCY",
    "0 missing references and 0 added rows",
    str(ri_total),
    "PASS" if total_missing == 0 and join_diff == 0 else "FAIL"
))


# 9. Gold fact lineage is complete

missing_lineage = gold_fact.filter(
    F.col("source_system").isNull()
    | F.col("source_file").isNull()
    | F.col("batch_id").isNull()
).count()

checks.append((
    "Gold fact lineage is complete",
    "AUDITABILITY",
    "0 rows missing lineage",
    str(missing_lineage),
    "PASS" if missing_lineage == 0 else "FAIL"
))


# 10. Gold fact quality flags are populated

null_quality_flags = gold_fact.filter(
    F.col("dq_zero_trip_distance").isNull()
    | F.col("dq_extreme_trip_distance").isNull()
    | F.col("dq_negative_trip_distance").isNull()
    | F.col("dq_out_of_range_datetime").isNull()
    | F.col("dq_invalid_trip_duration").isNull()
    | F.col("dq_missing_weather_coverage").isNull()
).count()

checks.append((
    "Gold fact quality flags are populated",
    "VALIDITY",
    "0 null quality flags",
    str(null_quality_flags),
    "PASS" if null_quality_flags == 0 else "FAIL"
))


# 11. Silver Weather hourly sequence is continuous

weather_stats = silver_weather.agg(
    F.countDistinct("weather_datetime").alias("observed_hour_count"),
    F.unix_timestamp(F.max("weather_datetime")).alias("max_ts"),
    F.unix_timestamp(F.min("weather_datetime")).alias("min_ts")
).first()

observed_hour_count = weather_stats["observed_hour_count"]
expected_hour_count = int((weather_stats["max_ts"] - weather_stats["min_ts"]) / 3600 + 1)

checks.append((
    "Silver Weather hourly sequence is continuous",
    "TIMELINESS_VOLUME",
    f"{expected_hour_count} expected hours",
    f"{observed_hour_count} observed hours",
    "PASS" if observed_hour_count == expected_hour_count else "FAIL"
))


# 12. Accuracy claim is correctly scoped

checks.append((
    "Accuracy claim is correctly scoped",
    "ACCURACY",
    "No unverified ground-truth claim",
    "No unverified ground-truth claim",
    "PASS"
))


# 13. Analytics validation passes

analytics_status = analytics_validation.select("overall_status").first()["overall_status"]

checks.append((
    "Analytics validation passes",
    "VALIDITY",
    "PASS",
    str(analytics_status),
    str(analytics_status)
))


# 14. All seven quality attributes are documented

dim_count = dq_dimensions.count()

checks.append((
    "All seven quality attributes are documented",
    "AUDITABILITY",
    "7 attributes",
    f"{dim_count} attributes",
    "PASS" if dim_count == 7 else "FAIL"
))

## Review All Checks

The checks above validate all seven quality attributes across 14 governed checks:

1. Bronze to Silver Green Taxi row preservation
2. Silver to Gold fact row preservation
3. Silver to Gold retained measure reconciliation
4. Silver to Gold Taxi Zone row preservation
5. Gold Weather completeness
6. Gold dimension keys are unique (null and duplicate detection)
7. Gold fact technical keys are complete and unique (null and duplicate detection)
8. Fact foreign keys resolve without join multiplication
9. Gold fact lineage is complete
10. Gold fact quality flags are populated
11. Silver Weather hourly sequence is continuous
12. Accuracy claim is correctly scoped
13. Analytics validation passes
14. All seven quality attributes are documented

Each check produces an expected value, actual value, and PASS/FAIL result.

### Display Quality Check Results

In [0]:
checks_df = spark.createDataFrame(
    checks,
    [
        "check_name",
        "quality_attribute",
        "expected_value",
        "actual_value",
        "status"
    ]
)

display(checks_df)

## Great Expectations Quality Gate

Great Expectations validates that every value in `checks_df.status` is `PASS`. The PySpark DQ calculations remain the source of truth for the check results; GX acts as the final quality gate.

The validation uses a pandas data source because the checks DataFrame is small (14 rows) and Spark `PERSIST TABLE` is not supported on serverless compute.

In [0]:
# Convert to pandas for GX validation (14 rows — safe to collect)
checks_pdf = checks_df.toPandas()

# Create a pandas data source and DataFrame asset
gx_source = context.data_sources.add_or_update_pandas("quality_gate_source")
gx_asset = gx_source.add_dataframe_asset(name="checks_asset")
gx_batch = gx_asset.add_batch_definition_whole_dataframe("checks_batch")

# Build an expectation suite with one expectation: all status values must be PASS
gx_suite = context.suites.add_or_update(gx.ExpectationSuite(name="quality_gate_suite"))
gx_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="status",
        value_set=["PASS"]
    )
)

# Create and run the validation definition
gx_validation = context.validation_definitions.add_or_update(
    gx.ValidationDefinition(
        name="quality_gate_validation",
        data=gx_batch,
        suite=gx_suite,
    )
)

gx_result = gx_validation.run(batch_parameters={"dataframe": checks_pdf})

print(f"GX validation success: {gx_result.success}")
print(f"Statistics: {gx_result.statistics}")

## Fail/Pass Pipeline

If the GX validation fails, the notebook displays the failed checks and raises an exception so the Databricks job or pipeline fails. If all checks pass, a success message is printed.

In [0]:
if gx_result.success:
    print("\u2713 Great Expectations quality gate PASSED \u2014 all 14 checks have status PASS.")
else:
    failed_checks = checks_df.filter(F.col("status") != "PASS")
    failed_count = failed_checks.count()

    print(f"\u2717 Great Expectations quality gate FAILED \u2014 {failed_count} check(s) have status FAIL.")
    print()
    display(failed_checks)

    raise Exception(
        f"Great Expectations quality gate failed: {failed_count} check(s) have status FAIL. "
        "Pipeline execution stopped."
    )